# 🏢 Sistema de Gestión de Recursos Humanos
## Módulo 1 — Reclutamiento
---
> **Flujo:** Keywords Binder → Carga de CVs → Extracción de texto → Análisis → Filtrado → Reporte

---
## Sección 0 — Instalación de dependencias

In [ ]:
# Instala todas las dependencias necesarias para el módulo
# Ejecuta esta celda una sola vez por entorno

import subprocess, sys

packages = [
    "pypdf2",       # Lectura de PDFs
    "python-docx",  # Lectura de DOCX
    "odfpy",        # Lectura de ODT
    "pandas",       # Manejo de tablas y exportación CSV
    "ipywidgets",   # Widgets interactivos en el notebook
    "IPython",      # Display y widgets
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Todas las dependencias instaladas correctamente.")

---
## Sección 1 — Importación de librerías

In [ ]:
import os
import re
import shutil
import csv
import zipfile
import pandas as pd
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output, HTML

# --- Lectura de PDF ---
try:
    import PyPDF2
except ImportError:
    PyPDF2 = None

# --- Lectura de DOCX ---
try:
    from docx import Document as DocxDocument
except ImportError:
    DocxDocument = None

# --- Lectura de ODT ---
try:
    from odf.opendocument import load as odf_load
    from odf.text import P
    from odf.teletype import extractText
except ImportError:
    odf_load = None

print("✅ Librerías importadas correctamente.")
print(f"   PDF   → {'✅ PyPDF2' if PyPDF2 else '❌ PyPDF2 no disponible'}")
print(f"   DOCX  → {'✅ python-docx' if DocxDocument else '❌ python-docx no disponible'}")
print(f"   ODT   → {'✅ odfpy' if odf_load else '❌ odfpy no disponible'}")

---
## Sección 2 — Keywords Binder

El archivo `.keyb` almacena las palabras clave del puesto en texto plano UTF-8, separadas por punto y coma (`;`).

**Opciones disponibles:**
1. 📂 Buscar un archivo `.keyb` local (en `/local_keywords_binder_files`)
2. 📤 Subir un archivo `.keyb` desde tu computadora
3. ✏️ Crear un nuevo archivo `.keyb`

In [ ]:
# ─────────────────────────────────────────────────
# 2.1 — Directorio base de Keywords Binder
# ─────────────────────────────────────────────────

KEYB_DIR = Path("local_keywords_binder_files")
KEYB_DIR.mkdir(exist_ok=True)

# Variable global que almacenará las keywords cargadas
keywords_activas = []
keyb_archivo_activo = ""

print(f"📁 Directorio de Keywords Binder: {KEYB_DIR.resolve()}")

In [ ]:
# ─────────────────────────────────────────────────
# 2.2 — Función: Crear nuevo archivo .keyb
# ─────────────────────────────────────────────────

def crear_keyb():
    """Guía al usuario para crear un nuevo archivo .keyb desde cero."""
    global keywords_activas, keyb_archivo_activo

    out = widgets.Output()
    keywords_temp = []

    titulo = widgets.HTML("<h4>✏️ Crear nuevo Keywords Binder</h4>")
    
    # Campo para agregar palabras clave
    input_kw   = widgets.Text(placeholder="Escribe una palabra clave y presiona Agregar",
                               layout=widgets.Layout(width="400px"))
    btn_add    = widgets.Button(description="➕ Agregar", button_style="info")
    btn_remove = widgets.Button(description="🗑️ Quitar última", button_style="warning")
    lista_kw   = widgets.Textarea(value="", placeholder="Palabras clave agregadas...",
                                   disabled=True, layout=widgets.Layout(width="400px", height="120px"))

    # Campo para nombre del archivo
    sep1       = widgets.HTML("<hr><b>Nombre del archivo</b> (sin espacios, sin extensión):")
    input_name = widgets.Text(placeholder="ej: ingeniero_de_datos",
                               layout=widgets.Layout(width="300px"))
    btn_save   = widgets.Button(description="💾 Guardar .keyb", button_style="success")
    btn_cancel = widgets.Button(description="Cancelar", button_style="danger")

    def on_add(b):
        kw = input_kw.value.strip()
        if kw and kw not in keywords_temp:
            keywords_temp.append(kw)
            lista_kw.value = " ; ".join(keywords_temp)
            input_kw.value = ""
        elif kw in keywords_temp:
            with out: print(f"⚠️  '{kw}' ya está en la lista.")

    def on_remove(b):
        if keywords_temp:
            removed = keywords_temp.pop()
            lista_kw.value = " ; ".join(keywords_temp)
            with out: print(f"🗑️  Se quitó: '{removed}'")

    def on_save(b):
        global keywords_activas, keyb_archivo_activo
        nombre = input_name.value.strip().replace(" ", "_")
        if not nombre:
            with out: print("❌ El nombre del archivo no puede estar vacío.")
            return
        if not keywords_temp:
            with out: print("❌ Debes agregar al menos una palabra clave.")
            return
        ruta = KEYB_DIR / f"{nombre}.keyb"
        contenido = ";".join(keywords_temp)
        ruta.write_text(contenido, encoding="utf-8")
        keywords_activas = keywords_temp.copy()
        keyb_archivo_activo = str(ruta)
        with out:
            clear_output()
            print(f"✅ Archivo guardado: {ruta}")
            print(f"📌 Keywords activas ({len(keywords_activas)}): {keywords_activas}")

    def on_cancel(b):
        with out:
            clear_output()
            print("❎ Creación cancelada.")

    btn_add.on_click(on_add)
    btn_remove.on_click(on_remove)
    btn_save.on_click(on_save)
    btn_cancel.on_click(on_cancel)

    display(titulo,
            widgets.HBox([input_kw, btn_add, btn_remove]),
            lista_kw,
            sep1, input_name,
            widgets.HBox([btn_save, btn_cancel]),
            out)

print("✅ Función crear_keyb() lista.")

In [ ]:
# ─────────────────────────────────────────────────
# 2.3 — Función: Buscar .keyb locales
# ─────────────────────────────────────────────────

def buscar_keyb_local():
    """Lista y permite seleccionar archivos .keyb en /local_keywords_binder_files."""
    global keywords_activas, keyb_archivo_activo

    archivos = list(KEYB_DIR.glob("*.keyb"))
    out = widgets.Output()

    if not archivos:
        display(widgets.HTML(
            "<p>⚠️ No se encontraron archivos <code>.keyb</code> en "
            f"<code>{KEYB_DIR}</code>. Crea uno nuevo o sube uno desde tu computadora.</p>"
        ))
        return

    opciones = {f.name: f for f in archivos}
    selector = widgets.Select(
        options=list(opciones.keys()),
        description="Archivo:",
        layout=widgets.Layout(width="350px", height="150px")
    )
    btn_cargar = widgets.Button(description="📂 Cargar seleccionado", button_style="success")
    preview    = widgets.Textarea(value="", placeholder="Vista previa del archivo...",
                                   disabled=True, layout=widgets.Layout(width="400px", height="80px"))

    def on_change(change):
        ruta = opciones[selector.value]
        contenido = ruta.read_text(encoding="utf-8")
        preview.value = contenido

    def on_cargar(b):
        global keywords_activas, keyb_archivo_activo
        ruta = opciones[selector.value]
        contenido = ruta.read_text(encoding="utf-8")
        keywords_activas = [k.strip() for k in contenido.split(";") if k.strip()]
        keyb_archivo_activo = str(ruta)
        with out:
            clear_output()
            print(f"✅ Cargado: {ruta.name}")
            print(f"📌 Keywords activas ({len(keywords_activas)}): {keywords_activas}")

    selector.observe(on_change, names="value")
    btn_cargar.on_click(on_cargar)

    # Mostrar preview del primer elemento
    if archivos:
        preview.value = archivos[0].read_text(encoding="utf-8")

    display(widgets.HTML("<h4>📂 Selecciona un archivo Keywords Binder local</h4>"),
            selector, preview, btn_cargar, out)

print("✅ Función buscar_keyb_local() lista.")

In [ ]:
# ─────────────────────────────────────────────────
# 2.4 — Función: Subir .keyb desde computadora
# ─────────────────────────────────────────────────

def subir_keyb():
    """Permite subir un archivo .keyb desde la computadora del usuario."""
    global keywords_activas, keyb_archivo_activo

    out = widgets.Output()
    uploader = widgets.FileUpload(
        accept=".keyb",
        multiple=False,
        description="Subir .keyb"
    )
    btn_procesar = widgets.Button(description="✅ Procesar archivo", button_style="success")

    def on_procesar(b):
        global keywords_activas, keyb_archivo_activo
        if not uploader.value:
            with out: print("❌ No se ha subido ningún archivo.")
            return
        # Compatibilidad con diferentes versiones de ipywidgets
        item = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        nombre  = item["metadata"]["name"] if isinstance(item, dict) else item.name
        content = item["content"] if isinstance(item, dict) else item.content.tobytes()
        if isinstance(content, (bytes, memoryview)):
            content = bytes(content).decode("utf-8")
        ruta_destino = KEYB_DIR / nombre
        ruta_destino.write_text(content, encoding="utf-8")
        keywords_activas = [k.strip() for k in content.split(";") if k.strip()]
        keyb_archivo_activo = str(ruta_destino)
        with out:
            clear_output()
            print(f"✅ Archivo guardado en: {ruta_destino}")
            print(f"📌 Keywords activas ({len(keywords_activas)}): {keywords_activas}")

    btn_procesar.on_click(on_procesar)
    display(widgets.HTML("<h4>📤 Subir archivo .keyb desde tu computadora</h4>"),
            uploader, btn_procesar, out)

print("✅ Función subir_keyb() lista.")

In [ ]:
# ─────────────────────────────────────────────────
# 2.5 — Menú principal del Keywords Binder
# ─────────────────────────────────────────────────

def menu_keywords_binder():
    """Menú principal con las 3 opciones del Keywords Binder."""
    out_menu = widgets.Output()

    titulo = widgets.HTML("""
        <h3>🔑 Keywords Binder</h3>
        <p>Selecciona cómo deseas cargar las palabras clave del puesto:</p>
    """)
    btn_buscar = widgets.Button(description="📂 Buscar local",  button_style="info",    layout=widgets.Layout(width="180px"))
    btn_subir  = widgets.Button(description="📤 Subir archivo", button_style="warning", layout=widgets.Layout(width="180px"))
    btn_crear  = widgets.Button(description="✏️ Crear nuevo",   button_style="success", layout=widgets.Layout(width="180px"))

    def on_buscar(b):
        with out_menu:
            clear_output()
            buscar_keyb_local()

    def on_subir(b):
        with out_menu:
            clear_output()
            subir_keyb()

    def on_crear(b):
        with out_menu:
            clear_output()
            crear_keyb()

    btn_buscar.on_click(on_buscar)
    btn_subir.on_click(on_subir)
    btn_crear.on_click(on_crear)

    display(titulo,
            widgets.HBox([btn_buscar, btn_subir, btn_crear]),
            out_menu)

# ── EJECUCIÓN ──
menu_keywords_binder()

---
## Sección 3 — Importación de CVs

Formatos soportados: `pdf`, `docx`, `doc`, `odt`, `txt`

**Opciones disponibles:**
1. 📤 Subir CVs desde tu computadora
2. 📂 Buscar CVs en una carpeta local

In [ ]:
# ─────────────────────────────────────────────────
# 3.1 — Directorio temporal para CVs cargados
# ─────────────────────────────────────────────────

CV_INPUT_DIR = Path("cv_input_temp")
CV_INPUT_DIR.mkdir(exist_ok=True)

FORMATOS_VALIDOS = {".pdf", ".docx", ".doc", ".odt", ".txt"}

# Lista global de rutas de CVs cargados
cvs_cargados = []

print(f"📁 Directorio temporal de CVs: {CV_INPUT_DIR.resolve()}")

In [ ]:
def subir_cvs():
    """Permite subir múltiples CVs desde la computadora."""
    global cvs_cargados

    out = widgets.Output()
    uploader = widgets.FileUpload(
        accept=".pdf,.docx,.doc,.odt,.txt",
        multiple=True,
        description="Subir CVs"
    )
    btn_procesar = widgets.Button(description="Procesar CVs", button_style="success")

    def on_procesar(b):
        global cvs_cargados
        if not uploader.value:
            with out: print("❌ No se han subido archivos.")
            return

        val = uploader.value
        # Normalizar a lista de (nombre, contenido_bytes)
        pares = []
        if isinstance(val, dict):
            for nombre, item in val.items():
                pares.append((nombre, bytes(item["content"])))
        elif isinstance(val, (list, tuple)):
            for item in val:
                if hasattr(item, "name"):
                    pares.append((item.name, bytes(item.content)))
                elif isinstance(item, dict):
                    pares.append((item["name"], bytes(item["content"])))

        nuevos = []
        rechazados = []

        for nombre, content in pares:
            ext = Path(nombre).suffix.lower()
            if ext not in FORMATOS_VALIDOS:
                rechazados.append(nombre)
                continue
            ruta_destino = CV_INPUT_DIR / nombre
            ruta_destino.write_bytes(content)
            nuevos.append(ruta_destino)

        cvs_cargados.extend(nuevos)

        with out:
            clear_output()
            print(f"✅ {len(nuevos)} CV(s) cargado(s) correctamente.")
            for r in nuevos: print(f"   📄 {r.name}")
            if rechazados:
                print(f"\n⚠️  {len(rechazados)} archivo(s) rechazado(s) por formato no soportado:")
                for r in rechazados: print(f"   ❌ {r}")
            print(f"\n📋 Total CVs en cola: {len(cvs_cargados)}")

    btn_procesar.on_click(on_procesar)
    display(widgets.HTML("<h4>Subir CVs desde tu computadora</h4>"),
            uploader, btn_procesar, out)

In [ ]:
# ─────────────────────────────────────────────────
# 3.3 — Función: Buscar CVs en carpeta local
# ─────────────────────────────────────────────────

def buscar_cvs_local():
    """Escanea una ruta local ingresada por el usuario y carga los CVs válidos."""
    global cvs_cargados

    out = widgets.Output()
    input_ruta = widgets.Text(
        placeholder="Ruta de la carpeta con CVs (ej: C:/Users/Juan/CVs)",
        layout=widgets.Layout(width="450px")
    )
    btn_escanear = widgets.Button(description="🔍 Escanear carpeta", button_style="info")
    btn_cargar   = widgets.Button(description="✅ Cargar CVs encontrados", button_style="success",
                                   disabled=True)
    resultado    = widgets.Textarea(value="", placeholder="CVs encontrados...",
                                    disabled=True, layout=widgets.Layout(width="450px", height="150px"))
    rutas_encontradas = []

    def on_escanear(b):
        ruta_str = input_ruta.value.strip()
        carpeta  = Path(ruta_str)
        rutas_encontradas.clear()
        if not carpeta.exists() or not carpeta.is_dir():
            with out: print(f"❌ La ruta no existe o no es una carpeta: {ruta_str}")
            return
        for f in carpeta.iterdir():
            if f.suffix.lower() in FORMATOS_VALIDOS:
                rutas_encontradas.append(f)
        if rutas_encontradas:
            resultado.value = "\n".join([r.name for r in rutas_encontradas])
            btn_cargar.disabled = False
            with out:
                clear_output()
                print(f"🔍 Se encontraron {len(rutas_encontradas)} CV(s).")
        else:
            resultado.value = "No se encontraron CVs válidos."
            with out:
                clear_output()
                print("⚠️  No se encontraron archivos con formatos válidos en esa carpeta.")

    def on_cargar(b):
        global cvs_cargados
        cvs_cargados.extend(rutas_encontradas)
        with out:
            clear_output()
            print(f"✅ {len(rutas_encontradas)} CV(s) agregado(s) a la cola.")
            print(f"📋 Total CVs en cola: {len(cvs_cargados)}")
        btn_cargar.disabled = True

    btn_escanear.on_click(on_escanear)
    btn_cargar.on_click(on_cargar)

    display(widgets.HTML("<h4>📂 Buscar CVs en carpeta local</h4>"),
            input_ruta, btn_escanear, resultado, btn_cargar, out)

print("✅ Función buscar_cvs_local() lista.")

In [ ]:
# ─────────────────────────────────────────────────
# 3.4 — Menú principal de carga de CVs
# ─────────────────────────────────────────────────

def menu_carga_cvs():
    """Menú para elegir cómo cargar los CVs."""
    out_menu = widgets.Output()

    titulo   = widgets.HTML("""
        <h3>📄 Carga de CVs</h3>
        <p>Formatos soportados: <code>pdf, docx, doc, odt, txt</code></p>
        <p>Selecciona cómo deseas cargar los CVs:</p>
    """)
    btn_subir  = widgets.Button(description="📤 Subir desde PC",     button_style="info",    layout=widgets.Layout(width="200px"))
    btn_buscar = widgets.Button(description="📂 Buscar carpeta local", button_style="warning", layout=widgets.Layout(width="200px"))

    def on_subir(b):
        with out_menu:
            clear_output()
            subir_cvs()

    def on_buscar(b):
        with out_menu:
            clear_output()
            buscar_cvs_local()

    btn_subir.on_click(on_subir)
    btn_buscar.on_click(on_buscar)

    display(titulo,
            widgets.HBox([btn_subir, btn_buscar]),
            out_menu)

# ── EJECUCIÓN ──
menu_carga_cvs()

---
## Sección 4 — Extracción de texto

In [ ]:
# ─────────────────────────────────────────────────
# 4.1 — Extractores por formato
# ─────────────────────────────────────────────────

def extraer_texto_pdf(ruta: Path) -> str:
    """Extrae texto de un archivo PDF."""
    if PyPDF2 is None:
        return ""
    texto = []
    try:
        with open(ruta, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    texto.append(t)
    except Exception as e:
        print(f"⚠️  Error leyendo PDF '{ruta.name}': {e}")
    return " ".join(texto)


def extraer_texto_docx(ruta: Path) -> str:
    """Extrae texto de un archivo DOCX."""
    if DocxDocument is None:
        return ""
    try:
        doc = DocxDocument(str(ruta))
        return " ".join([p.text for p in doc.paragraphs if p.text])
    except Exception as e:
        print(f"⚠️  Error leyendo DOCX '{ruta.name}': {e}")
        return ""


def extraer_texto_doc(ruta: Path) -> str:
    """Extrae texto de un archivo DOC legacy (intenta conversión temporal a DOCX)."""
    # Intento 1: usando python-docx directamente (funciona en algunos .doc)
    try:
        return extraer_texto_docx(ruta)
    except Exception:
        pass
    # Intento 2: leyendo como binario y extrayendo texto legible
    try:
        with open(ruta, "rb") as f:
            raw = f.read()
        # Extrae cadenas legibles de al menos 4 caracteres
        texto = re.findall(rb'[\x20-\x7E]{4,}', raw)
        return " ".join([t.decode("latin-1", errors="ignore") for t in texto])
    except Exception as e:
        print(f"⚠️  Error leyendo DOC '{ruta.name}': {e}")
        return ""


def extraer_texto_odt(ruta: Path) -> str:
    """Extrae texto de un archivo ODT."""
    if odf_load is None:
        return ""
    try:
        doc = odf_load(str(ruta))
        parrafos = doc.getElementsByType(P)
        return " ".join([extractText(p) for p in parrafos])
    except Exception as e:
        print(f"⚠️  Error leyendo ODT '{ruta.name}': {e}")
        return ""


def extraer_texto_txt(ruta: Path) -> str:
    """Lee un archivo de texto plano."""
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return ruta.read_text(encoding=enc)
        except (UnicodeDecodeError, Exception):
            continue
    return ""


print("✅ Extractores por formato listos.")

In [ ]:
# ─────────────────────────────────────────────────
# 4.2 — Función maestra de extracción
# ─────────────────────────────────────────────────

def extraer_texto(ruta: Path) -> str:
    """
    Detecta el formato del archivo y llama al extractor correspondiente.
    Retorna el texto completo en minúsculas para normalizar la búsqueda.
    """
    ext = ruta.suffix.lower()
    extractores = {
        ".pdf":  extraer_texto_pdf,
        ".docx": extraer_texto_docx,
        ".doc":  extraer_texto_doc,
        ".odt":  extraer_texto_odt,
        ".txt":  extraer_texto_txt,
    }
    fn = extractores.get(ext)
    if fn is None:
        print(f"⚠️  Formato no soportado: {ext}")
        return ""
    return fn(ruta).lower()


print("✅ Función maestra extraer_texto() lista.")

---
## Sección 5 — Análisis y puntuación de CVs

In [ ]:
# ─────────────────────────────────────────────────
# 5.1 — Umbrales de clasificación
# ─────────────────────────────────────────────────

UMBRAL_HIGH = 80.1   # Porcentaje mínimo para high_accuracy
UMBRAL_LOW  = 55.0   # Porcentaje mínimo para low_accuracy
# < UMBRAL_LOW → descartado

print(f"⚙️  Umbrales configurados:")
print(f"   🟢 high_accuracy : >= {UMBRAL_HIGH}%")
print(f"   🟡 low_accuracy  : {UMBRAL_LOW}% – {UMBRAL_HIGH}%")
print(f"   🔴 descartado    : < {UMBRAL_LOW}%")

In [ ]:
# ─────────────────────────────────────────────────
# 5.2 — Función: analizar un CV contra las keywords
# ─────────────────────────────────────────────────

def analizar_cv(ruta: Path, keywords: list) -> dict:
    """
    Extrae el texto del CV y lo compara contra las keywords.
    
    Retorna un diccionario con:
        - archivo           : nombre del CV
        - palabras_encontradas : lista de keywords halladas
        - palabras_faltantes   : lista de keywords no halladas
        - coincidencias     : cantidad de keywords encontradas
        - total_keywords    : total de keywords del .keyb
        - porcentaje        : % de coincidencia
        - estado            : high_accuracy | low_accuracy | descartado
        - ruta              : objeto Path al archivo original
    """
    texto = extraer_texto(ruta)

    encontradas = []
    faltantes   = []

    for kw in keywords:
        # Búsqueda insensible a mayúsculas
        if kw.lower() in texto:
            encontradas.append(kw)
        else:
            faltantes.append(kw)

    total      = len(keywords)
    coincid    = len(encontradas)
    porcentaje = round((coincid / total * 100), 2) if total > 0 else 0.0

    if porcentaje >= UMBRAL_HIGH:
        estado = "high_accuracy"
    elif porcentaje >= UMBRAL_LOW:
        estado = "low_accuracy"
    else:
        estado = "descartado"

    return {
        "archivo":              ruta.name,
        "palabras_encontradas": ", ".join(encontradas),
        "palabras_faltantes":   ", ".join(faltantes),
        "coincidencias":        coincid,
        "total_keywords":       total,
        "porcentaje":           porcentaje,
        "estado":               estado,
        "ruta":                 ruta,
    }


print("✅ Función analizar_cv() lista.")

In [ ]:
# ─────────────────────────────────────────────────
# 5.3 — Función: analizar todos los CVs
# ─────────────────────────────────────────────────

resultados_analisis = []  # Variable global con todos los resultados

def analizar_todos_los_cvs():
    """Corre el análisis sobre todos los CVs cargados y guarda los resultados."""
    global resultados_analisis

    if not cvs_cargados:
        print("❌ No hay CVs cargados. Ve a la Sección 3 y carga al menos un CV.")
        return
    if not keywords_activas:
        print("❌ No hay keywords activas. Ve a la Sección 2 y carga un Keywords Binder.")
        return

    print(f"⚙️  Analizando {len(cvs_cargados)} CV(s) con {len(keywords_activas)} keyword(s)...\n")
    resultados_analisis = []

    for ruta in cvs_cargados:
        res = analizar_cv(ruta, keywords_activas)
        resultados_analisis.append(res)
        icono = {"high_accuracy": "🟢", "low_accuracy": "🟡", "descartado": "🔴"}.get(res["estado"], "⚪")
        print(f"  {icono} {res['archivo']:40s} → {res['porcentaje']:6.2f}%  [{res['estado']}]")

    # Ordenar de mayor a menor porcentaje
    resultados_analisis.sort(key=lambda x: x["porcentaje"], reverse=True)

    high  = sum(1 for r in resultados_analisis if r["estado"] == "high_accuracy")
    low   = sum(1 for r in resultados_analisis if r["estado"] == "low_accuracy")
    desc  = sum(1 for r in resultados_analisis if r["estado"] == "descartado")

    print(f"\n{'─'*55}")
    print(f"  🟢 high_accuracy : {high}")
    print(f"  🟡 low_accuracy  : {low}")
    print(f"  🔴 descartados   : {desc}")
    print(f"  📋 Total          : {len(resultados_analisis)}")
    print(f"{'─'*55}")
    print("✅ Análisis completo. Continúa con la Sección 6.")


# ── EJECUCIÓN ──
analizar_todos_los_cvs()

---
## Sección 6 — Filtrado y organización de carpetas

In [ ]:
# ─────────────────────────────────────────────────
# 6.1 — Crear estructura de carpetas filtered_cv
# ─────────────────────────────────────────────────

FILTERED_DIR      = Path("filtered_cv")
HIGH_ACCURACY_DIR = FILTERED_DIR / "high_accuracy"
LOW_ACCURACY_DIR  = FILTERED_DIR / "low_accuracy"

HIGH_ACCURACY_DIR.mkdir(parents=True, exist_ok=True)
LOW_ACCURACY_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Estructura de carpetas creada:")
print(f"   {FILTERED_DIR.resolve()}")
print(f"   ├── high_accuracy/")
print(f"   └── low_accuracy/")

In [ ]:
# ─────────────────────────────────────────────────
# 6.2 — Copiar CVs clasificados a sus carpetas
# ─────────────────────────────────────────────────

def filtrar_y_copiar_cvs():
    """Copia los CVs high y low accuracy a sus carpetas. Los descartados no se copian."""
    if not resultados_analisis:
        print("❌ No hay resultados de análisis. Ejecuta la Sección 5 primero.")
        return

    copiados_high = 0
    copiados_low  = 0
    errores       = []

    for res in resultados_analisis:
        if res["estado"] == "descartado":
            continue

        destino_dir = HIGH_ACCURACY_DIR if res["estado"] == "high_accuracy" else LOW_ACCURACY_DIR
        origen      = res["ruta"]
        destino     = destino_dir / origen.name

        try:
            shutil.copy2(str(origen), str(destino))
            if res["estado"] == "high_accuracy":
                copiados_high += 1
                print(f"  🟢 → high_accuracy/{origen.name}")
            else:
                copiados_low += 1
                print(f"  🟡 → low_accuracy/{origen.name}")
        except Exception as e:
            errores.append((origen.name, str(e)))
            print(f"  ❌ Error copiando '{origen.name}': {e}")

    print(f"\n{'─'*55}")
    print(f"  🟢 Copiados a high_accuracy : {copiados_high}")
    print(f"  🟡 Copiados a low_accuracy  : {copiados_low}")
    print(f"  🔴 Descartados (no copiados): {sum(1 for r in resultados_analisis if r['estado'] == 'descartado')}")
    if errores:
        print(f"  ⚠️  Errores al copiar       : {len(errores)}")
    print(f"{'─'*55}")
    print("✅ Filtrado completado. Continúa con la Sección 7.")


# ── EJECUCIÓN ──
filtrar_y_copiar_cvs()

---
## Sección 7 — Generación del reporte

In [ ]:
# ─────────────────────────────────────────────────
# 7.1 — Exportar reporte .csv
# ─────────────────────────────────────────────────

def exportar_reporte_csv():
    """Genera y guarda el reporte CSV con todos los CVs analizados."""
    if not resultados_analisis:
        print("❌ No hay resultados. Ejecuta la Sección 5 primero.")
        return None

    columnas = ["archivo", "coincidencias", "total_keywords",
                "porcentaje", "estado", "palabras_encontradas", "palabras_faltantes"]

    ruta_csv = FILTERED_DIR / "reporte_reclutamiento.csv"

    with open(ruta_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columnas)
        writer.writeheader()
        for res in resultados_analisis:
            writer.writerow({col: res[col] for col in columnas})

    print(f"✅ Reporte exportado: {ruta_csv.resolve()}")
    return ruta_csv


ruta_reporte = exportar_reporte_csv()

In [ ]:
# ─────────────────────────────────────────────────
# 7.2 — Visualizar tabla resumen en el notebook
# ─────────────────────────────────────────────────

def mostrar_tabla_resumen():
    """Muestra la tabla de resultados con colores según el estado."""
    if not resultados_analisis:
        print("❌ No hay resultados que mostrar.")
        return

    columnas = ["archivo", "coincidencias", "total_keywords",
                "porcentaje", "estado", "palabras_encontradas", "palabras_faltantes"]

    df = pd.DataFrame([{col: r[col] for col in columnas} for r in resultados_analisis])

    def colorear_estado(val):
        colores = {
            "high_accuracy": "background-color: #d4edda; color: #155724; font-weight: bold",
            "low_accuracy":  "background-color: #fff3cd; color: #856404; font-weight: bold",
            "descartado":    "background-color: #f8d7da; color: #721c24; font-weight: bold",
        }
        return colores.get(val, "")

    def colorear_porcentaje(val):
        if val >= UMBRAL_HIGH:
            return "background-color: #d4edda"
        elif val >= UMBRAL_LOW:
            return "background-color: #fff3cd"
        else:
            return "background-color: #f8d7da"

    styled = (df.style
               .map(colorear_estado,     subset=["estado"])
               .map(colorear_porcentaje, subset=["porcentaje"])
               .format({"porcentaje": "{:.2f}%"})
               .set_caption("Reporte de Reclutamiento - Analisis de CVs")
               .set_table_styles([{
                   "selector": "caption",
                   "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left")]
               }]))

    display(styled)

    print(f"\nResumen:")
    print(f"   high_accuracy : {len(df[df.estado == 'high_accuracy'])} CV(s)")
    print(f"   low_accuracy  : {len(df[df.estado == 'low_accuracy'])} CV(s)")
    print(f"   descartados   : {len(df[df.estado == 'descartado'])} CV(s)")
    print(f"   Promedio de coincidencia: {df['porcentaje'].mean():.2f}%")
    print(f"   Mejor candidato: {df.iloc[0]['archivo']} ({df.iloc[0]['porcentaje']:.2f}%)")


# ── EJECUCIÓN ──
mostrar_tabla_resumen()

---
## Sección 8 — Ejecución completa (pipeline)

> Corre esta celda **después** de haber cargado tu Keywords Binder (Sección 2) y tus CVs (Sección 3).
> Ejecuta el pipeline completo de análisis → filtrado → reporte en una sola celda.

In [ ]:
# ─────────────────────────────────────────────────
# 8 — Pipeline completo de reclutamiento
# ─────────────────────────────────────────────────

def ejecutar_pipeline():
    """Ejecuta el pipeline completo: análisis → filtrado → reporte."""

    print("=" * 60)
    print("  🚀 PIPELINE — MÓDULO 1: RECLUTAMIENTO")
    print("=" * 60)

    # Validaciones previas
    if not keywords_activas:
        print("\n❌ DETENIDO: No hay keywords activas.")
        print("   → Ve a la Sección 2 y carga o crea un Keywords Binder.")
        return

    if not cvs_cargados:
        print("\n❌ DETENIDO: No hay CVs cargados.")
        print("   → Ve a la Sección 3 y carga los CVs.")
        return

    print(f"\n📌 Keywords activas    : {len(keywords_activas)} → {keywords_activas}")
    print(f"📄 CVs en cola         : {len(cvs_cargados)}")
    print(f"⚙️  Umbrales            : high >= {UMBRAL_HIGH}% | low >= {UMBRAL_LOW}%\n")

    # PASO 1: Análisis
    print("-" * 60)
    print("PASO 1 — Análisis y puntuación")
    print("-" * 60)
    analizar_todos_los_cvs()

    # PASO 2: Filtrado
    print("\n" + "-" * 60)
    print("PASO 2 — Filtrado y copia de CVs")
    print("-" * 60)
    filtrar_y_copiar_cvs()

    # PASO 3: Reporte CSV
    print("\n" + "-" * 60)
    print("PASO 3 — Exportación de reporte")
    print("-" * 60)
    exportar_reporte_csv()

    # PASO 4: Tabla visual
    print("\n" + "-" * 60)
    print("PASO 4 — Tabla resumen")
    print("-" * 60)
    mostrar_tabla_resumen()

    print("\n" + "=" * 60)
    print("  ✅ PIPELINE COMPLETADO")
    print(f"  📁 CVs filtrados en  : {FILTERED_DIR.resolve()}")
    print(f"  📊 Reporte en        : {(FILTERED_DIR / 'reporte_reclutamiento.csv').resolve()}")
    print("=" * 60)


# ── EJECUCIÓN ──
ejecutar_pipeline()